In [160]:
# https://www.kaggle.com/code/roylevy/pamap2-roy-yuval
# Bad Accuracy on PAMAP2 even with supervised
# https://github.com/kingdomrush2/CrossHAR

In [161]:
import os
import sys
from tqdm import tqdm
import numpy as np
import random
import torch
import torch.nn.functional as F
import torch.nn as nn
import itertools
import pandas as pd
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

In [162]:
torch.cuda.is_available()

True

In [163]:
torch.cuda.empty_cache()

In [164]:
base_url = os.getcwd()
sys.path.append(base_url + '/Util')

In [165]:
# from model import *
# from visualization import *
from datapreprocess import *
# from miscellaneous import *

# Intended Task

In [166]:
TASKS = ["Cross-Position", "Cross-Person", "Cross-Device"]
task = None
while True:
    print("1. Cross-Position\n2. Cross-Person\n3. Cross-Device")
    user_input = input("Please enter your task: ")
    if user_input == "1":
        task = "Cross-Position"
    elif user_input == "2":
        task = "Cross-Person"
    elif user_input == "3":
        task = "Cross-Device"
    if task in TASKS:
        break
    else:
        print("Invalid task. Please choose a valid task.")
print(f"Chosen task: {task}")

1. Cross-Position
2. Cross-Person
3. Cross-Device
Please enter your task: 3
Chosen task: Cross-Device


# Dataset

In [167]:
DATASETS = ["OPP", "DSADS", "PAMAP", "WISDM", "HHAR"]
dataset = None
if task == 'Cross-Position':
    while True:
        print("1. OPP\n2. DSADS\n3. PAMAP\n")
        user_input = input("Please enter your dataset: ")
        if user_input == "1":
            dataset = "OPP"
        elif user_input == "2":
            dataset = "DSADS"
        elif user_input == "3":
            dataset = "PAMAP"
            
        if dataset in DATASETS:
            break
        else:
            print("Invalid dataset. Please choose a valid dataset.")
elif task == 'Cross-Person':
    while True:
        print("1. OPP\n2. DSADS\n3. PAMAP\n4. WISDM\n")
        user_input = input("Please enter your dataset: ")
        if user_input == "1":
            dataset = "OPP"
        elif user_input == "2":
            dataset = "DSADS"
        elif user_input == "3":
            dataset = "PAMAP"
        elif user_input == "4":
            dataset = "WISDM"
            
        if dataset in DATASETS:
            break
        else:
            print("Invalid dataset. Please choose a valid dataset.")
elif task == 'Cross-Device':
    while True:
        print("1. WISDM\n")
        user_input = input("Please enter your dataset: ")
        if user_input == "1":
            dataset = "WISDM"
            
        if dataset in DATASETS:
            break
        else:
            print("Invalid dataset. Please choose a valid dataset.")
print(f"Chosen dataset: {dataset}")

1. WISDM

Please enter your dataset: 1
Chosen dataset: WISDM


# Reproduciblitiy (Must Change Before Each Experiment)

In [168]:
os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"

def set_seed(seed=42, loader=None):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    try:
        loader.sampler.generator.manual_seed(seed)
    except AttributeError:
        pass

seeds = [3431, 42, 1000]
seed = seeds[0] # Change this
set_seed(seed=seed)

# Experiment Setup

In [169]:
# All of these have to taken input from shell script
win_size = 120
overlap = 0

if task == "Cross-Position":
    if dataset == "DSADS":
        domains = {0:"TORSO", 1:"RA", 2:"LA", 3:"RL", 4:"LL"}
        sources = [0, 1, 2, 3]
        nsources = len(sources)
        target = 4
        activities = [ "standing", "lying-back", "ascending", "walking-parking-lot", "treadmill-running", "stepper-exercise", "cross-trainer-exercise", "rowing", "jumping",  "playing-basketball"]
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_list = ["User1","User2","User3","User4", "User5","User6","User7","User8"]
        position_array = ["TORSO","RA","LA","RL","LL"]
    elif dataset == "OPP":
        domains = {0:"BACK", 1:"RUA", 2:"RLA", 3:"LUA", 4:"LLA"}
        sources = [0, 1, 2, 3]
        nsources = len(sources)
        target = 4
        activities = ['Sitting','Standing','Walking','Running']
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_list = ["User1","User2","User3","User4"]
        position_array = ["BACK", "RUA", "RLA", "LUA","LLA"]
    elif dataset == "PAMAP":
        domains = {0:"Wrist", 1:"Chest", 2:"Ankle"}
        sources = [0, 1]
        nsources = len(sources)
        target = 2
        # activities = ['lying', 'sitting', 'standing', 'walking', 'running', 'vacuum', 'ironing', 'rope_jumping']
        activities = ['lying', 'sitting', 'standing', 'walking', 'vacuum', 'ironing']
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_list = ["User1","User2","User4","User5","User6","User7","User8"]
        position_array = ['Wrist', 'Chest', 'Ankle']
        
elif task == "Cross-Person":
    if dataset == "DSADS":
        domains = {0:"User1", 1:"User2", 2:"User3", 3:"User4", 4:"User5", 5:"User6", 6:"User7", 7:"User8"}
        sources = [0, 1, 2, 3, 5, 6, 7]
        target = [4]
        nsources = len(sources)
        position_list = ["TORSO", "RA", "LA", "RL", "LL"]
        activities = ["standing", "lying-back", "ascending", "walking-parking-lot", "treadmill-running", "stepper-exercise", "cross-trainer-exercise", "rowing", "jumping",  "playing-basketball"]
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_array = ["User1","User2","User3","User4","User5","User6","User7","User8"]
    elif dataset == "OPP":
        domains = {0:"User1", 1:"User2", 2:"User3", 3:"User4"}
        sources = [0, 2, 3]
        target = [1]
        nsources = len(sources)
        position_list = ["BACK", "RUA", "RLA", "LUA","LLA"]
        activities = ['Sitting','Standing','Walking','Running']
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_array = ["User1","User2","User3","User4"]
    elif dataset == "PAMAP":
        # domains = {0:'U1', 1:'U2', 2:'U3', 3:'U4', 4:'U5', 5:'U6', 6:'U7', 7:'U8'}
        domains = {0:'U1', 1:'U2', 2:'U4', 3:'U5', 4:'U6', 5:'U7', 6:'U8'}
        sources = [0, 1, 2, 3]
        nsources = len(sources)
        target = [4, 5, 6]
        activities = ['lying', 'sitting', 'standing', 'walking', 'vacuum', 'ironing']
        activity_num = len(activities)
        item = ["train","valid","test"]
        position_list = ['Wrist', 'Chest', 'Ankle']
        person_array = ["U1","U2","U4","U5","U6","U7","U8"]
    elif dataset == "WISDM":
        # -------------------------------------------------------------------- #
        # ----------------------- WISDM Cross-Person ------------------------- #
        # -------------------------------------------------------------------- #
        domains = {0:"U1", 1:"U5", 2:"U7", 3:"U8", 4:"U12", 5:"U13", 6:"U15", 7:"U37", 8:"U38", 9:"U39", 10:"U40"}
        sources = [0, 1, 2, 3, 4, 5, 6]
        target = [9]
        nsources = len(sources)
        position_list = ["phone", 'watch']
        # activities = ['walking', 'jogging', 'stairs', 'sitting', 'standing', 'kicking', 'catch', 'dribbling']
        activities = ['walking', 'jogging', 'stairs', 'sitting', 'standing', 'kicking', 'catch', 'dribbling']
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_array = ["U1","U5","U7","U8","U12","U13","U15","U37","U38","U39"]
elif task == "Cross-Device":
    if dataset == "WISDM":
        domains = {0:"phone", 1:"watch"}
        sources = [1]
        target = [0]
        nsources = len(sources)
        device_list = ["phone", "watch"]
        activities = ['walking', 'jogging', 'stairs', 'sitting', 'standing', 'kicking', 'catch', 'dribbling']
        activity_num = len(activities)
        item = ["train","valid","test"]
        # from user1 to user51.
        person_list = [f"U{i}" for i in range(1, 52)]
        # person_list_phone_reduced can be used for source=watch, target=phone
        # since watch has fewer samples compared to phone, it becomes difficult to perform DA
        # when all the users are considered. therefore, take fewer sample from the target->phone
        # person_list_phone_reduced = [f"U{i}" for i in range(1, 27)]
        ## yield better performance with a more balanced data distribution
        person_list_phone = ['U1', 'U5', 'U7', 'U8', 'U12', 'U13', 'U15', 'U16', 'U18', 'U23', 'U25', 'U31', 'U32', 'U33', 'U35', 'U37', 'U38', 'U39', 'U40', 'U41', 'U42', 'U43', 'U44', 'U45', 'U46', 'U47', 'U49', 'U51']
        person_list_watch = ['U2', 'U4', 'U5', 'U7', 'U11', 'U12', 'U14', 'U15', 'U16', 'U17', 'U19', 'U20', 'U23', 'U24', 'U28', 'U30', 'U31', 'U32', 'U33', 'U35', 'U37', 'U38', 'U39', 'U40', 'U41', 'U42', 'U44', 'U47', 'U49']
        person_map = {0:person_list_phone, 1:person_list_watch}
        # person_map = {0:person_list, 1:person_list}
        
print(activity_num)

8


In [170]:
step_size = int(win_size * (1 - overlap))
gpu_id = 0
DEVICE = torch.device('cuda:'+str(gpu_id) if torch.cuda.is_available() else 'cpu')

In [171]:
if dataset == "DSADS":
    AXIS = 6
    # for reading imu data
    FROM = 0
    TO = FROM+6
    # for activity label
    START = 7
    END = 8
elif dataset == "OPP":
    AXIS = 6
    # for reading imu data
    FROM = 0
    TO = FROM+6
    # for activity label
    START = 7
    END = 8
elif dataset == "PAMAP":
    AXIS = 6
    # for reading imu data
    FROM = 0
    TO = FROM+6
    # for activity label
    START = 7
    END = 8
elif dataset == "WISDM":
    AXIS = 6
    # for reading imu data
    FROM = 0
    TO = FROM+6
    # for activity label
    START = 7
    END = 8

In [172]:
if task == "Cross-Position":
    if dataset == "DSADS":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/dataset/DSADS_raw/"
        save_path = base_url + "/Output-v3/DSADS-Cross-Position-Heterogeneity/"+folder_name+"/Target-Position-" + position_array[target] + '/seed' + str(seed) + "/"
        main_folder = "DSADS-AllPerson-DifferentPosition"
    elif dataset == "OPP":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + '/dataset/OPPORTUNITY_raw/'
        save_path = base_url + "/Output-v3/OPP-Cross-Position-Heterogeneity/"+folder_name+"/Target-Position-" + position_array[target] + '/seed' + str(seed) + "/"
        main_folder = "OPP-AllPerson-DifferentPosition"
    elif dataset == "PAMAP":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/dataset/PAMAP2_raw/"
        save_path = base_url + "/Output-v3/PAMAP-Cross-Position-Heterogeneity/"+folder_name+"/Target-Position-" + position_array[target] + '/seed' + str(seed) + "/"
        main_folder = "PAMAP-AllPerson-DifferentPosition"
        
elif task == "Cross-Person":
    target_string = "_".join([domains[i] for i in target])
    if dataset == "DSADS":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/dataset/DSADS_raw/"
        save_path = base_url + "/Output-v3/DSADS-Cross-Person-Heterogeneity/"+folder_name+"/Target-" + target_string + '/seed' + str(seed) + "/"
        main_folder = "DSADS-SamePosition-DifferentPerson"
    elif dataset == "OPP":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/dataset/OPPORTUNITY_raw/"
        save_path = base_url + "/Output-v3/OPP-Cross-Person-Heterogeneity/"+folder_name+"/Target-" + target_string + '/seed' + str(seed) + "/"
        main_folder = "OPP-SamePosition-DifferentPerson"
    elif dataset == "PAMAP":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/Dataset_Preprocessed/PAMAP/Data Files/"+folder_name+"/"
        save_path = base_url + "/Output-v3/PAMAP-Cross-Person-Heterogeneity/"+folder_name+"/Target-" + target_string + '/seed' + str(seed) + "/"
        main_folder = "PAMAP-AllPerson-DifferentPerson"
    elif dataset == "WISDM":
        target_string = "_".join([domains[i] for i in target])
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/dataset/WISDM_raw/"
        save_path = base_url + "/Output/WISDM-Cross-Person-Heterogeneity/" + folder_name + "/Target-" + target_string + '/seed' + str(seed) + "/"
        main_folder = "WISDM-AllPerson-DifferentDevice"

elif task == "Cross-Device":
    target_string = "_".join([domains[i] for i in target])
    if dataset == "WISDM":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/dataset/WISDM_raw/"
        save_path = base_url + "/Output-v3/WISDM-Cross-Device-Heterogeneity/" + folder_name + "/Target-" + target_string + '/seed' + str(seed) + "/"
        main_folder = "WISDM-AllPerson-DifferentDevice"
        
if not os.path.exists(save_path):
    os.makedirs(save_path)

In [173]:
source_data = [[] for _ in range(nsources)]
target_data = []

source_gt = [[] for _ in range(nsources)]
target_gt = []

In [174]:
# sources can be [0, 1, 3, 4], and target = 2
print(f"Dataset: {dataset},\tTask: {task}")
if task == 'Cross-Position':
    print('Source')
    for i in range(len(sources)):
        # Create an empty DataFrame to hold the aggregated data for this source position
        df_aggregated_source = pd.DataFrame()

        # Combine data from all users for the current source position
        for person in person_list:
            file_name = person + "_" + position_array[sources[i]]
            print(f"Processing: {file_name}")
            df = pd.read_csv(dataset_path + file_name + '.csv', sep=",")
            df_aggregated_source = pd.concat([df_aggregated_source, df], ignore_index=True)
        # Pass the aggregated data to calculate_window
        calculate_window(df_aggregated_source, source_data[i], source_gt[i], win_size, step_size, FROM, TO, START, END, AXIS)

    # For the target position, similarly aggregate data across all users
    df_aggregated_target = pd.DataFrame()
    print('Target')
    for person in person_list:
        file_name = person + "_" + position_array[target]
        print(f"Processing: {file_name}")
        df = pd.read_csv(dataset_path + file_name + '.csv', sep=",")
        df_aggregated_target = pd.concat([df_aggregated_target, df], ignore_index=True)
    calculate_window(df_aggregated_target, target_data, target_gt, win_size, step_size, FROM, TO, START, END, AXIS)

        
elif task == 'Cross-Person':
    print('Source')
    for i in range(len(sources)):
        # Create an empty DataFrame to hold the aggregated data for this source position
        df_aggregated_source = pd.DataFrame()

        for position in position_list:
            file_name = person_array[sources[i]] + "_" + position
            print(f"Processing: {file_name}")
            df = pd.read_csv(dataset_path + file_name + '.csv', sep=",")
            df_aggregated_source = pd.concat([df_aggregated_source, df], ignore_index=True)

        # Pass the aggregated data to calculate_window
        calculate_window(df_aggregated_source, source_data[i], source_gt[i], win_size, step_size, FROM, TO, START, END, AXIS)

    # For the target position, similarly aggregate data across all users
    print('Target')
    df_aggregated_target = pd.DataFrame()
    for i in range(len(target)):
        for position in position_list:
            file_name = person_array[target[i]] + "_" + position
            print(f"Processing: {file_name}")
            df = pd.read_csv(dataset_path + file_name + '.csv', sep=",")
            df_aggregated_target = pd.concat([df_aggregated_target, df], ignore_index=True)
            
    calculate_window(df_aggregated_target, target_data, target_gt, win_size, step_size, FROM, TO, START, END, AXIS)

elif task == "Cross-Device" and dataset == "WISDM":
    print('Source')
    for i in range(len(sources)):
        # Create an empty DataFrame to hold the aggregated data for this source position
        df_aggregated_source = pd.DataFrame()

        for person in person_map[sources[0]]:
        #for person in person_list:
            file_name = person + "_" + device_list[sources[i]]
            print(f"Processing: {file_name}")
            df = pd.read_csv(dataset_path + file_name + '.csv', sep=",")
            df_aggregated_source = pd.concat([df_aggregated_source, df], ignore_index=True)
                    
        # Pass the aggregated data to calculate_window
        calculate_window(df_aggregated_source, source_data[i], source_gt[i], win_size, step_size, FROM, TO, START, END, AXIS)

    # For the target position, similarly aggregate data across all users
    print('Target')
    df_aggregated_target = pd.DataFrame()
    
    for i in range(len(target)):
        for person in person_map[target[0]]:
            file_name = person + "_" + device_list[target[i]]
            print(f"Processing: {file_name}")
            df = pd.read_csv(dataset_path + file_name + '.csv', sep=",")
            df_aggregated_target = pd.concat([df_aggregated_target, df], ignore_index=True)
                    
    # Pass the aggregated data to calculate_window
    calculate_window(df_aggregated_target, target_data, target_gt, win_size, step_size, FROM, TO, START, END, AXIS)

Dataset: WISDM,	Task: Cross-Device
Source
Processing: U2_watch
Processing: U4_watch
Processing: U5_watch
Processing: U7_watch
Processing: U11_watch
Processing: U12_watch
Processing: U14_watch
Processing: U15_watch
Processing: U16_watch
Processing: U17_watch
Processing: U19_watch
Processing: U20_watch
Processing: U23_watch
Processing: U24_watch
Processing: U28_watch
Processing: U30_watch
Processing: U31_watch
Processing: U32_watch
Processing: U33_watch
Processing: U35_watch
Processing: U37_watch
Processing: U38_watch
Processing: U39_watch
Processing: U40_watch
Processing: U41_watch
Processing: U42_watch
Processing: U44_watch
Processing: U47_watch
Processing: U49_watch
Target
Processing: U1_phone
Processing: U5_phone
Processing: U7_phone
Processing: U8_phone
Processing: U12_phone
Processing: U13_phone
Processing: U15_phone
Processing: U16_phone
Processing: U18_phone
Processing: U23_phone
Processing: U25_phone
Processing: U31_phone
Processing: U32_phone
Processing: U33_phone
Processing: U

In [175]:
# Number of rows or number of training domains (different positions)
print(type(source_data[0][0]))
print(len(source_data))
print(source_data[0][0].shape)

<class 'numpy.ndarray'>
1
(1, 120, 6)


In [176]:
source_flat = np.concatenate([np.squeeze(np.array(sublist), axis=1) for sublist in source_data], axis=0)
if dataset=='OPP' or dataset=='PAMAP' or dataset=="WISDM":
    source_flat = np.array(source_flat, dtype=np.float32)

In [177]:
source_flat

array([[[ 1.31903782e-01,  2.85589874e-01,  1.49558127e+00,
          4.13689196e-01, -2.91853309e-01, -4.75687414e-01],
        [ 6.62439093e-02,  1.93614051e-01,  1.66484928e+00,
          1.81907892e-01, -2.85370797e-01, -4.97282654e-01],
        [ 1.50142655e-01,  1.09108396e-01,  1.32631302e+00,
          2.06132069e-01, -1.73399776e-01, -4.73352790e-01],
        ...,
        [-9.43277001e-01,  7.54929259e-02, -1.33299008e-01,
          2.14907423e-01, -6.04716480e-01, -4.59930778e-01],
        [-1.26306486e+00, -7.71775171e-02, -1.48489714e-01,
          4.98990923e-01, -6.01769924e-01, -6.35610878e-01],
        [-1.02808774e+00,  2.08087444e-01, -2.21839264e-01,
          1.13322270e+00, -6.87808573e-01, -6.84054434e-01]],

       [[-7.51464963e-01,  4.26121384e-01, -3.13851655e-01,
          1.24223161e+00, -8.47514570e-01, -7.96699882e-01],
        [-5.83059430e-01,  4.56935614e-01, -3.74614567e-01,
          8.16656828e-01, -9.83647764e-01, -9.90473270e-01],
        [-4.82441

In [178]:
source_flat.shape

(7225, 120, 6)

In [179]:
# # Function to find all unique data types in source_flat
# def find_data_types(array):
#     data_types = set()

#     # Iterate through each element of the top-level list (array)
#     for subarray in array:
#         if isinstance(subarray, np.ndarray):
#             # If it's a numpy array, check each element inside this array
#             for inner_array in subarray:
#                 if isinstance(inner_array, np.ndarray):
#                     # If it's a nested numpy array, check its elements
#                     for item in inner_array:
#                         data_types.add(type(item))
#                 else:
#                     # Otherwise, just check the type of the element
#                     data_types.add(type(inner_array))
#         else:
#             # If it's not a numpy array, check the type of the element directly
#             data_types.add(type(subarray))

#     return data_types

# # Example usage
# data_types = find_data_types(source_flat)
# print("Unique data types in source_flat:", data_types)

In [180]:
# Save s_gt_train as an .npy file
if dataset=='DSADS':
    np.save("dataset/dsads_s/data_20_120.npy", source_flat)
elif dataset=='OPP':
    np.save("dataset/opportunity_s/data_20_120.npy", source_flat)
elif dataset=='PAMAP':
    np.save("dataset/pamap2_s/data_20_120.npy", source_flat)
elif dataset=='WISDM':
    np.save("dataset/wisdm_s/data_20_120.npy", source_flat)

In [181]:
if dataset=='DSADS':
    datanp = np.load("dataset/dsads_s/data_20_120.npy")
elif dataset=='OPP':
    datanp = np.load("dataset/opportunity_s/data_20_120.npy")
elif dataset=='PAMAP':
    datanp = np.load("dataset/pamap2_s/data_20_120.npy")
elif dataset=='WISDM':
    datanp = np.load("dataset/wisdm_s/data_20_120.npy")
print(datanp.shape)

(7225, 120, 6)


In [182]:
print(len(source_gt))
print(len(source_gt[0]))
# print(len(source_gt[1]))
# print(len(source_gt[2]))
print(len(source_gt))
print(type(source_gt))
print(type(source_gt[0]))
print(type(source_gt[0][0]))

1
7225
1
<class 'list'>
<class 'list'>
<class 'numpy.int64'>


In [183]:
source_gt_array = np.concatenate(source_gt)
source_gt_array = source_gt_array.reshape(-1, 1, 1)

In [184]:
source_gt_array.shape

(7225, 1, 1)

In [185]:
if dataset=='DSADS':
    np.save('dataset/dsads_s/label_20_120.npy', source_gt_array)
elif dataset=='OPP':
    np.save('dataset/opportunity_s/label_20_120.npy', source_gt_array)
elif dataset=='PAMAP':
    np.save("dataset/pamap2_s/label_20_120.npy", source_gt_array)
elif dataset=='WISDM':
    np.save("dataset/wisdm_s/label_20_120.npy", source_gt_array)

In [186]:
if dataset=='DSADS':
    labelnp = np.load("dataset/dsads_s/label_20_120.npy")
elif dataset=='OPP':
    labelnp = np.load("dataset/opportunity_s/label_20_120.npy")
elif dataset=='PAMAP':
    labelnp = np.load("dataset/pamap2_s/label_20_120.npy")
elif dataset=='WISDM':
    labelnp = np.load("dataset/wisdm_s/label_20_120.npy")
print(labelnp.shape)

(7225, 1, 1)


In [187]:
print(type(target_data))
print(len(target_data))
print(target_data[0].shape)

<class 'list'>
7251
(1, 120, 6)


In [188]:
target_data_stack = np.squeeze(np.stack(target_data))
if dataset=='OPP' or dataset=='PAMAP' or dataset=="WISDM":
    target_data_stack = np.array(target_data_stack, dtype=np.float32)
# Verify the shape of the resulting array
print(target_data_stack.shape)  # Should print (5000, 120, 6)

(7251, 120, 6)


In [189]:
if dataset=='DSADS':
    np.save("dataset/dsads_t/data_20_120.npy", target_data_stack)
elif dataset=='OPP':
    np.save("dataset/opportunity_t/data_20_120.npy", target_data_stack)
elif dataset=='PAMAP':
    np.save("dataset/pamap2_t/data_20_120.npy", target_data_stack)
elif dataset=='WISDM':
    np.save("dataset/wisdm_t/data_20_120.npy", target_data_stack)

In [190]:
# Assuming target_gt is a list of 5000 integers
target_gt_array = np.array(target_gt)
target_gt_array = target_gt_array.reshape(-1, 1, 1)
# Check if each element in the list corresponds to the shape (120, 6)
# We need to reshape the list to (5000, 120, 6)
# target_gt_reshaped = target_gt_array.reshape(5000, 120, 6)

# Verify the shape of the resulting array
print(target_gt_array.shape)  # Should print (5000, 120, 6)

# Save the array in npy format
if dataset=='DSADS':
    np.save("dataset/dsads_t/label_20_120.npy", target_gt_array)
elif dataset=='OPP':
    np.save("dataset/opportunity_t/label_20_120.npy", target_gt_array)
elif dataset=='PAMAP':
    np.save("dataset/pamap2_t/label_20_120.npy", target_gt_array)
elif dataset=='WISDM':
    np.save("dataset/wisdm_t/label_20_120.npy", target_gt_array)

(7251, 1, 1)


In [191]:
if dataset=='DSADS':
    datatargetnp = np.load("dataset/dsads_t/data_20_120.npy")
elif dataset=='OPP':
    datatargetnp = np.load("dataset/opportunity_t/data_20_120.npy")
elif dataset=='PAMAP':
    datatargetnp = np.load("dataset/pamap2_t/data_20_120.npy")
elif dataset=='WISDM':
    datatargetnp = np.load("dataset/wisdm_t/data_20_120.npy")
print(datatargetnp.shape)

(7251, 120, 6)


In [192]:
if dataset=='DSADS':
    labeltargetnp = np.load("dataset/dsads_t/label_20_120.npy")
elif dataset=='OPP':
    labeltargetnp = np.load("dataset/opportunity_t/label_20_120.npy")
elif dataset=='PAMAP':
    labeltargetnp = np.load("dataset/pamap2_t/label_20_120.npy")
elif dataset=='WISDM':
    labeltargetnp = np.load("dataset/wisdm_t/label_20_120.npy")
print(labeltargetnp.shape)

(7251, 1, 1)
